In [1]:
%%capture
!pip install -q -U datasets transformers torch

In [8]:
from datasets import load_dataset
from collections import Counter

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

## 1. Carregando a base de dados IMDB
O dataset IMDB contém 25.000 resenhas de filmes para treino e 25.000 para teste,
com rótulos binários: positivo (1) ou negativo (0).

In [9]:
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
train_ds = dataset["train"]
test_ds  = dataset["test"]

# Separa 20% do treino para validação
split      = train_ds.train_test_split(test_size=0.2, seed=42)
train_ds   = split["train"]
val_ds     = split["test"]

print(f"Treino:    {len(train_ds)}")
print(f"Validação: {len(val_ds)}")
print(f"Teste:     {len(test_ds)}")

print("\nDistribuição de rótulos (treino):", Counter(train_ds["label"]))
print("Distribuição de rótulos (teste): ", Counter(test_ds["label"]))

Treino:    20000
Validação: 5000
Teste:     25000

Distribuição de rótulos (treino): Counter({0: 10006, 1: 9994})
Distribuição de rótulos (teste):  Counter({0: 12500, 1: 12500})


In [5]:
# Exemplo de resenha e seu rótulo
print("Rótulo:", train_ds[0]["label"])
print("Texto:",  train_ds[0]["text"][:300], "...")

Rótulo: 1
Texto: Stage adaptations often have a major fault. They often come out looking like a film camera was simply placed on the stage (Such as "Night Mother"). Sidney Lumet's direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. Still, it's nice to look a ...


## 2. Configuração do Modelo
Utilizamos o **DistilBERT** (`distilbert-base-uncased`), uma versão compacta e
eficiente do BERT, adequada para classificação de sequências.

In [6]:
MODEL_ID      = "distilbert-base-uncased"
NUM_CLASSES   = 2
MAX_LEN       = 256
BATCH_SIZE    = 32
NUM_EPOCHS    = 3
OPTIMIZER     = "adamw_torch"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_CLASSES
)
print(model)
print(f"\nTotal de parâmetros: {model.num_parameters():,}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## 3. Tokenização
Convertemos os textos para tensores de IDs de tokens que o modelo aceita.

In [10]:
def tokenizar(exemplos):
    return tokenizer(
        exemplos["text"],
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True
    )

encoded_train = train_ds.map(tokenizar, batched=True, batch_size=BATCH_SIZE)
encoded_val   = val_ds.map(tokenizar,   batched=True, batch_size=BATCH_SIZE)
encoded_test  = test_ds.map(tokenizar,  batched=True, batch_size=BATCH_SIZE)

print("Dataset tokenizado:")
print(encoded_train)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Dataset tokenizado:
Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20000
})


## 4. Treinamento com `Trainer`
Usamos a API `Trainer` do HuggingFace para fazer o fine-tuning do modelo.

In [11]:
def calcular_metricas(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "acuracia": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds)
    }

In [12]:
training_args = TrainingArguments(
    output_dir="distilbert_imdb_checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    save_total_limit=1,
    greater_is_better=True,
    metric_for_best_model="f1",
    optim=OPTIMIZER,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    compute_metrics=calcular_metricas,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Acuracia,F1
1,0.222918,0.240559,0.899600,0.902600
2,0.162550,0.247055,0.902800,0.900776
3,0.050103,0.353633,0.906400,0.906137


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.17082668946584065, metrics={'train_runtime': 1552.9093, 'train_samples_per_second': 38.637, 'train_steps_per_second': 1.207, 'total_flos': 3974021959680000.0, 'train_loss': 0.17082668946584065, 'epoch': 3.0})

## 5. Avaliação no Conjunto de Teste

In [14]:
predictions = trainer.predict(encoded_test)
metricas_finais = calcular_metricas(
    (predictions.predictions, predictions.label_ids)
)
print("Resultados no conjunto de teste:")
print(f"  Acurácia: {metricas_finais['acuracia']:.4f}")
print(f"  F1-Score: {metricas_finais['f1']:.4f}")

Resultados no conjunto de teste:
  Acurácia: 0.9095
  F1-Score: 0.9096
